In [ ]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from flax import linen as nn
import jax_dataloader as jd

import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors

import torch
from torch.utils.data import Dataset

import os
from tqdm.auto import tqdm

#os.environ['CUDA_VISIBLE_DEVICES'] = '1'
config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true' # default is true, 90% of GPU VRAM preallocated

Poisson equation $\nabla^2 p = b$

The p and b data in cell_center_data is the exact same as the p and b in the parent dataset except the boundaries are stripped.

In this dataset, we should use Homogeneous Dirichlet Boundary Condition ($p = 0$).

$b$ is the source, $p$ is the pressure simulation ground truth, $\eta$ (eta) represents the geometric boundary mask of the solid obstacles where 0 means fluid, 1.0 means boundary.

This dataset is a simplier version of a bigger stokes flow dataset because we remove $u$ and $v$, where $b = \frac{\rho}{\Delta t} \nabla \cdot \mathbf{u}^*$

We solve the pressure Poisson equation here to predict $p$, meaning we want to learn the mapping $(b, \eta) \rightarrow p$ in a single forward pass.
For the physics loss function, we do $\nabla^2 p_{pred} = b$. For evaluation, we compare $p_{pred}$ with the simulation $p_{sim}$. We can add both into the loss function actually.

In [ ]:
T = 10
n_short = 10
n_gap = 20

DATA_DIR = Path.cwd().parent.parent / "data" / "poisson-solver" / "T10_data"

# 960x512
p_file = DATA_DIR / "cell_center_data" / "p" / f"dxdy_960x512_p_T{T}_S{n_short}_G{n_gap}.npy"
b_file = DATA_DIR / "cell_center_data" / "b" / f"dxdy_960x512_b_T{T}_S{n_short}_G{n_gap}.npy"

# 962x514
eta_file = DATA_DIR / "eta" / f"dxdy_960x512_eta_T{T}_S{n_short}_G{n_gap}.npy" # need to crop the boundaries eta_grid[1:-1, 1:-1]
# p_file = DATA_DIR / "p" / f"dxdy_960x512_p_T{T}_S{n_short}_G{n_gap}.npy"
# b_file = DATA_DIR / "b" / f"dxdy_960x512_b_T{T}_S{n_short}_G{n_gap}.npy"


In [ ]:
p_grid = np.load(p_file)
b_grid = np.load(b_file)
eta_grid = np.load(eta_file)

print(f"The shape of the raw p array is: {p_grid.shape}")
print(f"The shape of the raw b array is: {b_grid.shape}")
print(f"The shape of the raw eta array is: {eta_grid.shape}")

eta_grid = eta_grid[1:-1, 1:-1]
print(f"The shape of the raw eta array is: {eta_grid.shape}")

In [ ]:
L = 1.0e-6
l_x = L * 960
l_y = L * 512
ext = [0, l_x, 0, l_y]

fig = plt.figure(figsize=(16, 4))

ax1 = fig.add_subplot(1, 3, 1)
s_plot = b_grid.T
mesh1 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh1)
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Source (b)', fontsize='x-large')

ax2 = fig.add_subplot(1, 3, 2)
s_plot = p_grid.T
mesh2 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh2)
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Simulation (Pressure, p)', fontsize='x-large')

ax3 = fig.add_subplot(1, 3, 3)
s_plot = eta_grid.T
mesh3 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='binary', extent=ext, aspect=1.)
plt.colorbar(mesh3)
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Boundaries (\u03B7)', fontsize='x-large')

plt.tight_layout()
plt.show()

In [ ]:
T, S, G = 10, 10, 20
DATA_DIR = Path.cwd().parent.parent / "data" / "poisson-solver" / "T10_data"
eta_file = DATA_DIR / "eta" / f"dxdy_960x512_eta_T{T}_S{S}_G{G}.npy"

eta_grid = np.load(eta_file)

unique_vals = np.unique(eta_grid)
print(f"1. Unique values in eta grid: {unique_vals}")
if len(unique_vals) <= 2:
    print("Conclusion: Eta is a binary mask (e.g., 0 for fluid, 1 for solid).")

# Strip the ghost cells to match the physical 960x512 domain
eta_inner = eta_grid[1:-1, 1:-1]

# Find exactly where the sifter is located on the Y-axis
# We sum across the X-axis. Rows with fluid will sum to 0. The row with the sifter will have a high sum.
col_sums = np.sum(eta_inner, axis=0) 
sifter_y_index = np.argmax(col_sums)
print(f"\n2. The sifter is located at Y-index (row): {sifter_y_index}")

# Extract a 1D horizontal slice right through the sifter
# We fix the Y-index and pull all X-values
sifter_slice = eta_inner[:, sifter_y_index]

# Plot the 1D slice
plt.figure(figsize=(12, 2))
plt.plot(sifter_slice[:1000], drawstyle='steps-mid', linewidth=2)
plt.title(rf"1D Slice of $\eta$ (First 1000 cells) | Expected: S={S}, G={G}")
plt.xlabel(r"X Coordinate ($\mu m$)")
plt.ylabel(r"$\eta$ value")
plt.grid(True, axis='x', alpha=0.5)
plt.show()

In [ ]:
# Load and Pre-process all 192 samples
T = 10
s_vals = range(10, 34, 2)
g_vals = range(20, 52, 2)

L = 1.0e-6
x_coords = np.linspace(0, 960 * L, 960)
y_coords = np.linspace(0, 512 * L, 512)

X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')

# Store globally as JAX arrays for later use in the training loop
GLOBAL_X = jnp.array(X.reshape(-1, 1))
GLOBAL_Y = jnp.array(Y.reshape(-1, 1))

B_SCALE_FACTOR = 1e11

print("Loading .npy files into RAM...")
all_inputs = []
all_targets = []

for s in s_vals:
    for g in g_vals:
        p_file = DATA_DIR / "cell_center_data" / "p" / f"dxdy_960x512_p_T{T}_S{s}_G{g}.npy"
        b_file = DATA_DIR / "cell_center_data" / "b" / f"dxdy_960x512_b_T{T}_S{s}_G{g}.npy"
        eta_file = DATA_DIR / "eta" / f"dxdy_960x512_eta_T{T}_S{s}_G{g}.npy"
        
        if p_file.exists() and b_file.exists() and eta_file.exists():
            b_grid = np.load(b_file)
            p_grid = np.load(p_file)
            eta_grid = np.load(eta_file)
            
            # Slice eta to match the 960x512 physical domain
            eta_inner = eta_grid[1:-1, 1:-1]
            
            # Scale b down to roughly [-1, 1]
            b_normalized = b_grid / B_SCALE_FACTOR

            # Flatten spatial dimensions (960 * 512 = 491520 points)
            b_flat = b_normalized.reshape(-1, 1)
            eta_flat = eta_inner.reshape(-1, 1)
            p_flat = p_grid.reshape(-1, 1)
            
            # Stack only the 2 unique features: [scaled_b, raw_eta]
            inputs_combined = np.concatenate([b_flat, eta_flat], axis=-1)
            
            all_inputs.append(inputs_combined)
            all_targets.append(p_flat)

# Expected Shapes -> Inputs: (192, 491520, 2) | Targets: (192, 491520, 1)
all_inputs = np.stack(all_inputs)
all_targets = np.stack(all_targets)
print(f"Total dataset shape  | Inputs (b, eta): {all_inputs.shape} | Targets (p): {all_targets.shape}")

In [ ]:
class PoissonStokesDataset(Dataset):
    def __init__(self, source_data, solution_data):
        self.s = source_data
        self.u = solution_data

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        return self.s[idx], self.u[idx]

# 192 total samples
n_train = 160
n_val = 16
n_test = 16
batch_size_samples = 8

# shuffle samples
np.random.seed(42)
shuffled_indices = np.random.permutation(len(all_inputs))
all_inputs = all_inputs[shuffled_indices]
all_targets = all_targets[shuffled_indices]

train_dataset = PoissonStokesDataset(source_data=all_inputs[:n_train], solution_data=all_targets[:n_train])
val_dataset = PoissonStokesDataset(source_data=all_inputs[n_train : n_train + n_val], solution_data=all_targets[n_train : n_train + n_val])
test_dataset = PoissonStokesDataset(source_data=all_inputs[-n_test:], solution_data=all_targets[-n_test:])

train_dataloader = jd.DataLoader(train_dataset, backend='pytorch', batch_size=batch_size_samples, shuffle=True, drop_last=True, num_workers=0)
val_dataloader = jd.DataLoader(val_dataset, backend='pytorch', batch_size=batch_size_samples, shuffle=False, num_workers=0)
test_dataloader = jd.DataLoader(test_dataset, backend='pytorch', batch_size=batch_size_samples, shuffle=False, num_workers=0)

print("JAX DataLoaders ready!")

In [ ]:
M = 256 
hidden_layers = [128, 128, 128, 128] 
total_features = (2 * M) + sum(hidden_layers) # 512 + 512 = 1024

rff_key = jax.random.PRNGKey(99)
B_matrix_base = jax.random.normal(rff_key, (2, M)) 

class FeatureExtractor(nn.Module):
    hidden_layers: list
    B_matrix_base: jnp.ndarray 
    
    @nn.compact
    def __call__(self, x_in, y_in):
        self.param('raw_tik', nn.initializers.constant(-4.0), (1,))
        
        # T10 Physical Boundaries
        x_l, x_u = 0.0, 960.0 * 1e-6
        y_l, y_u = 0.0, 512.0 * 1e-6
        
        # Normalization to [-1, 1]
        x_norm = 2.0 * (x_in - x_l) / (x_u - x_l) - 1.0
        y_norm = 2.0 * (y_in - y_l) / (y_u - y_l) - 1.0
        
        # RFF on coordinates
        # We'll leave sigma as a static float for now, but you can wrap it in self.param() later if you want to tune it
        sigma = 3.0 
        v_coords = jnp.concatenate([x_norm, y_norm], axis=-1)
        proj = 2.0 * jnp.pi * jnp.dot(v_coords, self.B_matrix_base * sigma)
        h_rff = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)
        
        all_features = [h_rff]
        h = h_rff

        for size in self.hidden_layers:
            h = nn.Dense(size, kernel_init=nn.initializers.he_normal())(h)
            h = nn.tanh(h) 
            all_features.append(h)
            
        return jnp.concatenate(all_features, axis=-1)

model = FeatureExtractor(
    hidden_layers=hidden_layers, 
    B_matrix_base=B_matrix_base 
)

# Initialize params
key = jax.random.PRNGKey(0)
dummy_x = jnp.array([0.0])
dummy_y = jnp.array([0.0])
params = model.init(key, dummy_x, dummy_y)

In [ ]:
def get_f(params, x_val, y_val):
    return model.apply(params, x_val, y_val)

def get_f_dir(params, x_val, y_val):
    f_x = jacfwd(get_f, argnums=1)(params, x_val, y_val)
    f_y = jacfwd(get_f, argnums=2)(params, x_val, y_val)
    
    f_xx = jacfwd(jacfwd(get_f, argnums=1), argnums=1)(params, x_val, y_val)
    f_yy = jacfwd(jacfwd(get_f, argnums=2), argnums=2)(params, x_val, y_val)
    
    f_x = jnp.squeeze(f_x, axis=-1)
    f_y = jnp.squeeze(f_y, axis=-1)
    f_xx = jnp.squeeze(f_xx, axis=(-1, -2))
    f_yy = jnp.squeeze(f_yy, axis=(-1, -2))
    
    return f_x, f_y, f_xx, f_yy

f_spatial_vmap = vmap(get_f, in_axes=(None, 0, 0))
f_dir_spatial_vmap = vmap(get_f_dir, in_axes=(None, 0, 0))

In [ ]:
chunk_size = 256

def get_f_chunk(params, x_val, y_val, start_idx, end_idx):
    f = model.apply(params, x_val, y_val)
    return lax.dynamic_slice(f, (start_idx,), (end_idx - start_idx,))

def get_f_dir_chunk(params, x_val, y_val, start_idx, end_idx):
    # Only compute second derivatives for the Laplacian
    f_xx = jacfwd(jacfwd(get_f_chunk, argnums=1), argnums=1)(params, x_val, y_val, start_idx, end_idx)
    f_yy = jacfwd(jacfwd(get_f_chunk, argnums=2), argnums=2)(params, x_val, y_val, start_idx, end_idx)
    
    # Squeeze out the inner gradient dimensions to match expected shapes
    f_xx = jnp.squeeze(f_xx, axis=(-1, -2))
    f_yy = jnp.squeeze(f_yy, axis=(-1, -2))
    
    return f_xx, f_yy

f_chunk_spatial_vmap = vmap(get_f_chunk, in_axes=(None, 0, 0, None, None))
f_dir_chunk_spatial_vmap = vmap(get_f_dir_chunk, in_axes=(None, 0, 0, None, None))

In [ ]:
@jax.jit(static_argnames=['total_features', 'chunk_size'])
def update_direct_solve_batched(params, x_pde, y_pde, b_batch, x_wall, y_wall, total_features, chunk_size, tik_reg):
    A_chunks = []
    B_SCALE_FACTOR = 1e11
    
    for start_idx in range(0, total_features, chunk_size):
        end_idx = min(start_idx + chunk_size, total_features)
        
        # PDE Block (Scaled down by 1e11 to match b_batch)
        f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, end_idx)
        A_pde_chunk = (f_xx + f_yy) / B_SCALE_FACTOR
        
        # Dirichlet
        A_wall_chunk = f_chunk_spatial_vmap(params, x_wall, y_wall, start_idx, end_idx)
        
        A_chunk = jnp.vstack([A_pde_chunk, A_wall_chunk])
        A_chunks.append(A_chunk)
        
    # Full Shared A matrix -> shape: (N_pde + N_wall, total_features)
    A = jnp.hstack(A_chunks)
    
    """
    Previously b was (2000,1). Now, it is (batch_size, 2000), hence we need the transpose.
    """
    # --- BATCHED RIGHT HAND SIDE ---
    # b_batch shape: (batch_size, N_pde, 1) -> squeeze to (batch_size, N_pde) -> Transpose to (N_pde, batch_size)
    b_pde = jnp.squeeze(b_batch, axis=-1).T  
    
    # Wall targets are all 0 -> shape: (N_wall, batch_size)
    b_wall = jnp.zeros((x_wall.shape[0], b_pde.shape[1])) 
    
    # Stack targets -> shape: (N_pde + N_wall, batch_size)
    RHS = jnp.vstack([b_pde, b_wall]) 
    
    # --- BATCHED TIKHONOV SOLVE ---
    AtA = A.T @ A + tik_reg * jnp.eye(total_features)
    AtB = A.T @ RHS  # Shape: (total_features, batch_size)
    
    # Solve for w -> Shape: (total_features, batch_size)
    w_batch_T = jnp.linalg.solve(AtA, AtB) 
    
    # Transpose to return a standard batch shape: (batch_size, total_features)
    return w_batch_T.T

In [ ]:
def full_forward_and_loss(params, x_pde, y_pde, b_batch, p_sim_batch, x_wall, y_wall, total_features, chunk_size):
    raw_tik = params['params']['raw_tik'][0]
    tik_reg = 10.0 ** raw_tik

    # 1. Get Batched Weights -> Shape: (batch_size, total_features)
    w_batch = update_direct_solve_batched(params, x_pde, y_pde, b_batch, x_wall, y_wall, total_features, chunk_size, tik_reg)

    # 2. Compute shared basis functions and Laplacians -> Shape: (N_points, total_features)
    f_pde = f_spatial_vmap(params, x_pde, y_pde)
    
    # We only need f_xx and f_yy for the PDE loss
    _, _, f_xx, f_yy = f_dir_spatial_vmap(params, x_pde, y_pde)
    
    # 3. Batched Predictions via Matrix Multiplication
    # f_pde is (N_points, features). w_batch.T is (features, batch_size).
    # Result is (N_points, batch_size). We transpose to get (batch_size, N_points).
    p_pred_batch = jnp.dot(f_pde, w_batch.T).T
    
    # Scale the Laplacian to match the scaled b_batch
    laplacian_pred_batch = jnp.dot((f_xx + f_yy) / 1e11, w_batch.T).T
    
    # Squeeze targets to match prediction shapes
    p_sim_batch_sq = jnp.squeeze(p_sim_batch, axis=-1)
    b_batch_sq = jnp.squeeze(b_batch, axis=-1)
    
    # 4. Calculate Losses
    loss_sim = jnp.mean((p_pred_batch - p_sim_batch_sq)**2)
    loss_pde = jnp.mean((laplacian_pred_batch - b_batch_sq)**2)
    
    # Wall Loss
    f_wall = f_spatial_vmap(params, x_wall, y_wall)
    p_wall_batch = jnp.dot(f_wall, w_batch.T).T
    loss_wall = jnp.mean(p_wall_batch**2)
    
    total_loss = loss_sim + loss_pde + loss_wall
    
    return total_loss, (w_batch, loss_sim, loss_pde, loss_wall, tik_reg)

# JIT compile the gradient function
loss_grad_fn = jax.jit(
    jax.value_and_grad(full_forward_and_loss, argnums=0, has_aux=True), 
    static_argnames=['total_features', 'chunk_size']
)

optimizer = optax.adamw(learning_rate=1e-3, weight_decay=1e-4)

@jax.jit(static_argnames=['total_features', 'chunk_size'])
def update_network(params, opt_state, x_pde, y_pde, b_batch, p_sim_batch, x_wall, y_wall, total_features, chunk_size):
    (loss, (w_batch, loss_sim, loss_pde, loss_wall, tik_reg)), grads = loss_grad_fn(
        params, x_pde, y_pde, b_batch, p_sim_batch, x_wall, y_wall, total_features, chunk_size
    )
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    new_params = optax.apply_updates(params, updates)
    
    return new_params, opt_state, loss, w_batch, loss_sim, loss_pde, loss_wall, tik_reg

In [ ]:
def get_batched_importance_indices(b_batch_chunk, threshold=1e-4):
    """
    Calculates active and empty indices for a batched spatial grid.
    b_batch_chunk shape: (batch_size, N_points, 1)
    """
    # Take the maximum absolute value across the batch dimension
    # This ensures if a pixel is active in ANY sample in the batch, it gets flagged
    b_max_across_batch = jnp.max(jnp.abs(b_batch_chunk), axis=0).flatten()
    
    source_indices = jnp.where(b_max_across_batch > threshold)[0]
    empty_indices = jnp.where(b_max_across_batch <= threshold)[0]
    
    return source_indices, empty_indices

In [ ]:
# Setup outer wall coordinates once
wall_x_mask = (GLOBAL_X == 0.0) | (GLOBAL_X == 960.0 * 1e-6)
wall_y_mask = (GLOBAL_Y == 0.0) | (GLOBAL_Y == 512.0 * 1e-6)
wall_mask = (wall_x_mask | wall_y_mask).flatten()

global_x_wall = GLOBAL_X[wall_mask]
global_y_wall = GLOBAL_Y[wall_mask]

@jax.jit
def fast_eval_chunk(params, w_batch, x_chunk, y_chunk):
    """Evaluate a spatial chunk for the whole batch."""
    f_eval = f_spatial_vmap(params, x_chunk, y_chunk) 
    # Predict pressure -> Shape: (batch_size, chunk_size)
    p_pred_batch = jnp.dot(f_eval, w_batch.T).T
    return p_pred_batch

def evaluate_dataset(dataloader, params):
    """Calculates the average MSE across an entire dataloader using full-grid chunking."""
    total_mse = 0.0
    total_samples = 0
    
    raw_tik = params['params']['raw_tik'][0]
    tik_reg = 10.0 ** raw_tik
    
    val_rng = jax.random.PRNGKey(100)
    chunk_size_eval = 10000 
    total_points = GLOBAL_X.shape[0]
    
    for inputs_batch, targets_batch in dataloader:
        b_batch = jnp.array(inputs_batch[:, :, 0:1])
        p_sim_batch = jnp.array(targets_batch)
        
        # --- 1. IMPORTANCE SAMPLING FOR THE SOLVER ---
        b_max_across_batch = jnp.max(jnp.abs(b_batch), axis=0).flatten()
        source_indices = jnp.where(b_max_across_batch > 1e-4)[0]
        empty_indices = jnp.where(b_max_across_batch <= 1e-4)[0]
        
        num_actual_sources = source_indices.shape[0]
        val_rng, s_key, e_key, wall_key = jax.random.split(val_rng, 4)
        
        n_sources_to_sample = min(1000, num_actual_sources)
        s_idx = jax.random.choice(s_key, source_indices, shape=(n_sources_to_sample,), replace=False)
        e_idx = jax.random.choice(e_key, empty_indices, shape=(2000 - n_sources_to_sample,), replace=False)
        
        solver_idx = jnp.concatenate([s_idx, e_idx])
        
        x_solver = GLOBAL_X[solver_idx]
        y_solver = GLOBAL_Y[solver_idx]
        b_solver = b_batch[:, solver_idx, :]
        
        wall_idx = jax.random.choice(wall_key, global_x_wall.shape[0], shape=(400,), replace=False)
        x_wall_val = global_x_wall[wall_idx]
        y_wall_val = global_y_wall[wall_idx]
        
        # Solve for w specifically for this validation batch
        w_val = update_direct_solve_batched(
            params, x_solver, y_solver, b_solver, 
            x_wall_val, y_wall_val, 
            total_features, chunk_size, tik_reg
        )
        
        # --- 2. FULL GRID EVALUATION IN CHUNKS ---
        u_pred_list = []
        
        for i in range(0, total_points, chunk_size_eval):
            x_c = GLOBAL_X[i : i+chunk_size_eval]
            y_c = GLOBAL_Y[i : i+chunk_size_eval]
            
            # Get predictions for this chunk across the whole batch
            p_pred_chunk = fast_eval_chunk(params, w_val, x_c, y_c)
            u_pred_list.append(p_pred_chunk)
            
        # Stack chunks horizontally to reconstruct full grid predictions
        # Final shape: (batch_size, 491520)
        p_pred_batch = jnp.hstack(u_pred_list) 
        
        # Squeeze true simulation pressure to match: (batch_size, 491520)
        p_sim_batch_sq = jnp.squeeze(p_sim_batch, axis=-1)
        
        batch_mse = jnp.mean((p_pred_batch - p_sim_batch_sq)**2)
        
        current_batch_size = b_batch.shape[0]
        total_mse += batch_mse.item() * current_batch_size
        total_samples += current_batch_size
            
    return total_mse / total_samples

In [ ]:
epochs = 100
spatial_batch_size = 2000
wall_batch_size = 400

opt_state = optimizer.init(params)
rng = jax.random.PRNGKey(42)

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for batch_idx, (inputs_batch, targets_batch) in enumerate(train_dataloader):
        # Move PyTorch tensors to JAX arrays
        b_batch = jnp.array(inputs_batch[:, :, 0:1]) 
        p_sim_batch = jnp.array(targets_batch)
        
        # --- 1. DYNAMIC IMPORTANCE SAMPLING ---
        # Find active source indices across the batch
        b_max_across_batch = jnp.max(jnp.abs(b_batch), axis=0).flatten()
        source_indices = jnp.where(b_max_across_batch > 1e-4)[0]
        empty_indices = jnp.where(b_max_across_batch <= 1e-4)[0]
        
        num_actual_sources = source_indices.shape[0]
        
        rng, s_key, e_key, wall_key = jax.random.split(rng, 4)
        
        # Sample up to 1000 active points, fill the rest with empty points
        n_sources_to_sample = min(1000, num_actual_sources)
        s_idx = jax.random.choice(s_key, source_indices, shape=(n_sources_to_sample,), replace=False)
        
        n_empty_to_sample = spatial_batch_size - n_sources_to_sample
        e_idx = jax.random.choice(e_key, empty_indices, shape=(n_empty_to_sample,), replace=False)
        
        pde_idx = jnp.concatenate([s_idx, e_idx])
        
        x_pde_chunk = GLOBAL_X[pde_idx]
        y_pde_chunk = GLOBAL_Y[pde_idx]
        
        # Slice the batched targets
        b_batch_chunk = b_batch[:, pde_idx, :]
        p_sim_batch_chunk = p_sim_batch[:, pde_idx, :]
        # --------------------------------------
        
        # 2. Randomly sample wall points
        total_wall_points = global_x_wall.shape[0]
        wall_idx = jax.random.choice(wall_key, total_wall_points, shape=(wall_batch_size,), replace=False)
        
        x_wall_chunk = global_x_wall[wall_idx]
        y_wall_chunk = global_y_wall[wall_idx]
        
        # 3. Perform Batched Update
        params, opt_state, loss, w_batch, l_sim, l_pde, l_wall, tik = update_network(
            params, opt_state, 
            x_pde_chunk, y_pde_chunk, b_batch_chunk, p_sim_batch_chunk, 
            x_wall_chunk, y_wall_chunk, 
            total_features, chunk_size
        )
        
        epoch_loss += loss
        
    # --- END OF EPOCH EVALUATION ---
    avg_train_loss = epoch_loss / len(train_dataloader)
    
    # Only run the heavy full-grid validation every 10 epochs
    if (epoch + 1) % 10 == 0:
        val_mse = evaluate_dataset(val_dataloader, params)
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4e} | Val MSE: {val_mse:.4e} | Tik: {tik:.2e}")
    else:
        # Print standard training metrics for the other epochs
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4e} | Tik: {tik:.2e}")

In [ ]:
# 1. Extract a single test sample
sample_idx = 0 
inputs_test, p_sim_test = test_dataset[sample_idx]

# Extract b and add a batch dimension so it plays nicely with our batched functions
b_test = inputs_test[:, 0:1]
b_batch_test = jnp.array(b_test[None, ...])
p_sim_batch_test = jnp.array(p_sim_test[None, ...])

# 2. Importance Sampling for the Test Solver
b_flat_test = jnp.abs(b_batch_test[0, :, 0])
source_indices = jnp.where(b_flat_test > 1e-4)[0]
empty_indices = jnp.where(b_flat_test <= 1e-4)[0]

num_actual_sources = source_indices.shape[0]

rng_test = jax.random.PRNGKey(123)
rng_test, s_key, e_key, wall_key = jax.random.split(rng_test, 4)

n_sources_to_sample = min(1000, num_actual_sources)
s_idx = jax.random.choice(s_key, source_indices, shape=(n_sources_to_sample,), replace=False)
e_idx = jax.random.choice(e_key, empty_indices, shape=(2000 - n_sources_to_sample,), replace=False)

solver_idx = jnp.concatenate([s_idx, e_idx])

x_solver = GLOBAL_X[solver_idx]
y_solver = GLOBAL_Y[solver_idx]
b_solver = b_batch_test[:, solver_idx, :]

# Sample validation wall points
wall_idx = jax.random.choice(wall_key, global_x_wall.shape[0], shape=(400,), replace=False)
x_wall_test = global_x_wall[wall_idx]
y_wall_test = global_y_wall[wall_idx]

raw_tik = params['params']['raw_tik'][0]
tik_reg = 10.0 ** raw_tik

# 3. Solve for w_test
print("Solving for w_test using importance sampling...")
w_test = update_direct_solve_batched(
    params, x_solver, y_solver, b_solver, 
    x_wall_test, y_wall_test, 
    total_features, chunk_size, tik_reg
)

# 4. Full Grid Evaluation
print("Evaluating full grid in chunks...")
chunk_size_eval = 10000 
total_points = GLOBAL_X.shape[0]
u_pred_list = []

for i in range(0, total_points, chunk_size_eval):
    x_c = GLOBAL_X[i : i+chunk_size_eval]
    y_c = GLOBAL_Y[i : i+chunk_size_eval]
    
    p_pred_chunk = fast_eval_chunk(params, w_test, x_c, y_c)
    u_pred_list.append(p_pred_chunk)
    
# Reconstruct the grid and remove the batch dimension
p_pred_batch = jnp.hstack(u_pred_list) 
p_pred = p_pred_batch[0] 

# Calculate Metrics
p_sim_flat = p_sim_test.flatten()
mse = jnp.mean((p_sim_flat - p_pred)**2)
mae = jnp.mean(jnp.abs(p_sim_flat - p_pred))
rl2 = jnp.linalg.norm(p_sim_flat - p_pred) / jnp.linalg.norm(p_sim_flat)

print(f"Test Metrics -> MSE: {mse:.2e} | MAE: {mae:.2e} | RL2: {rl2:.2e}")

# 5. Plotting
print("Generating plots...")
b_plot = b_test.reshape(960, 512).T
p_sim_plot = p_sim_test.reshape(960, 512).T
p_pred_plot = p_pred.reshape(960, 512).T

ext = [0, 960 * 1e-6, 0, 512 * 1e-6]

fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(1, 3, 1)
mesh1 = ax1.imshow(b_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh1, ax=ax1)
ax1.set_xlabel('x (m)')
ax1.set_ylabel('y (m)')
ax1.set_title('Test Source (b)', fontsize='x-large')

ax2 = fig.add_subplot(1, 3, 2)
mesh2 = ax2.imshow(p_sim_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh2, ax=ax2)
ax2.set_xlabel('x (m)')
ax2.set_ylabel('y (m)')
ax2.set_title('Simulation Ground Truth (p)', fontsize='x-large')

ax3 = fig.add_subplot(1, 3, 3)
mesh3 = ax3.imshow(p_pred_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh3, ax=ax3)
ax3.set_xlabel('x (m)')
ax3.set_ylabel('y (m)')
ax3.set_title('PINN Prediction', fontsize='x-large')

plt.tight_layout()
plt.show()